# Group 9 — SMS Spam Detection: Performance–Efficiency Trade-off

**Approved primary question:** Do the classification performance gains of fine-tuned DistilBERT justify its additional training and inference costs relative to frozen MiniLM embeddings and TF-IDF?

**Protocol:** fixed stratified 70/15/15 splits; tune on development only; evaluate held-out test once; full-pipeline timing from raw SMS text; report hardware, repetitions, batch size and tuning costs. Natural text-speak is a secondary exploratory analysis. Comparing different encoder architectures does **not** isolate the effect of fine-tuning.

## Setup

In [1]:
%pip install -q requests scikit-learn sentence-transformers "transformers>=4.44,<5" datasets accelerate torch pandas joblib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os, sys, time, platform, json, random, statistics, importlib.metadata as im
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix
from IPython.display import display
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS={p:im.version(p) for p in ['torch','scikit-learn','transformers','sentence-transformers','datasets','numpy']}
HARDWARE={'platform':platform.platform(),'python':sys.version.split()[0], 'cpu':platform.processor() or 'not reported', 'logical_cpu_count':os.cpu_count(), 'device':DEVICE, 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'versions':VERSIONS}
print(json.dumps(HARDWARE,indent=2))

{
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "python": "3.13.16",
  "cpu": "x86_64",
  "logical_cpu_count": 2,
  "device": "cuda",
  "gpu": "Tesla T4",
  "versions": {
    "torch": "2.14.1+cu130",
    "scikit-learn": "1.6.1",
    "transformers": "4.57.6",
    "sentence-transformers": "5.7.0",
    "datasets": "4.8.5",
    "numpy": "2.4.6"
  }
}


## Data: public UCI SMS Spam Collection, exact deduplication, stratified splits

In [3]:

import pandas as pd
import zipfile
import io
import requests
from sklearn.model_selection import train_test_split

# Download the official UCI SMS Spam Collection
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("SMSSpamCollection") as f:
        df = pd.read_csv(
            f,
            sep="\t",
            header=None,
            names=["label", "text"],
            encoding="utf-8"
        )

# Clean data
df = df.dropna(subset=["label", "text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df = df[df["label"].isin(["ham", "spam"])]
df = df[df["text"].ne("")]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Train / Development / Test split (70/15/15)
train, tmp = train_test_split(
    df, test_size=0.30,
    stratify=df["label"], random_state=SEED
)

dev, test = train_test_split(
    tmp, test_size=0.50,
    stratify=tmp["label"], random_state=SEED
)

assert not (set(train.text) & set(test.text))
assert not (set(train.text) & set(dev.text))
assert not (set(dev.text) & set(test.text))

Xtr, Xdev, Xte = (
    train.text.tolist(),
    dev.text.tolist(),
    test.text.tolist()
)

ytr = (train.label == "spam").astype(int).to_numpy()
yd = (dev.label == "spam").astype(int).to_numpy()
yt = (test.label == "spam").astype(int).to_numpy()

print("Class counts:")
print(df.label.value_counts())
print("Train:", len(train))
print("Dev:", len(dev))
print("Test:", len(test))


Class counts:
label
ham     4516
spam     642
Name: count, dtype: int64
Train: 3610
Dev: 774
Test: 774


## Timing protocol — instructor requirement

**Full-pipeline training:** TF-IDF vectorizer fit + classifier fit; MiniLM embedding extraction for **training** texts + classifier fit; DistilBERT tokenization of **training** texts + final fine-tuning. Pretrained model/tokenizer downloads and imports are excluded for all methods. Development-set preprocessing, hyperparameter search and epoch selection are **tuning costs**, reported separately, not folded into final training.

**Full-pipeline inference:** raw test SMS → TF-IDF transform / MiniLM embeddings / DistilBERT tokenization → model predictions. Run on the **same held-out 774 SMS**, in the same environment, with batch size **32** for neural models; median of **5** timed full-test runs after **1 warm-up**; synchronize GPU before/after. Inference includes preprocessing and prediction but excludes model loading and metric computation. Timings are **batch throughput** (total seconds / test count), **not single-message interactive latency**. CPU/GPU and library versions are recorded above.

**Caveat:** Training steps differ across architectures and are compared as complete practical pipelines, not as an isolated causal effect of fine-tuning. GPU/CPU availability affects times. Record new measurements after running this notebook; previous results must not be reused as new timings.

In [4]:
def sync():
    if torch.cuda.is_available():torch.cuda.synchronize()
def measure(fn, repeats=1, warmup=0):
    for _ in range(warmup):fn()
    times=[];out=None
    for _ in range(repeats):
        sync();start=time.perf_counter();out=fn();sync();times.append(time.perf_counter()-start)
    return out, float(np.median(times)), times
def metrics(p):
    return dict(accuracy=accuracy_score(yt,p),macro_f1=f1_score(yt,p,average='macro'),spam_recall=recall_score(yt,p,pos_label=1,zero_division=0))
results={};preds={};tuning_records=[];timing_details={}

## Method 1 — TF-IDF + Logistic Regression

In [5]:
def fit_tfidf(c):
    return make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=2),LogisticRegression(C=c,max_iter=1500,class_weight='balanced',random_state=SEED)).fit(Xtr,ytr)
t0=time.perf_counter()
tf_scores=[(f1_score(yd,fit_tfidf(c).predict(Xdev),average='macro'),c) for c in [.1,1,10]]
c=max(tf_scores)[1]
tuning_records.append({'model':'TF-IDF','tuning_s':time.perf_counter()-t0,'selection':f'C={c}','dev_macro_f1':max(tf_scores)[0]})
m1,ttrain,_=measure(lambda:fit_tfidf(c))
p,tinf,all_times=measure(lambda:m1.predict(Xte),repeats=5,warmup=1)
preds['TF-IDF']=p
results['TF-IDF']={**metrics(p),'train_s':ttrain,'inference_s':tinf,'ms_per_sms':1000*tinf/len(Xte)}
timing_details['TF-IDF']=all_times

## Method 2 — Frozen MiniLM + Logistic Regression

In [6]:
from sentence_transformers import SentenceTransformer
enc=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=DEVICE)
def embed(x):return enc.encode(x,batch_size=32,show_progress_bar=False,convert_to_numpy=True)
# Exclude pretrained model download/loading; measure embedding training texts and classifier fit.
Etr,train_embed_s,_=measure(lambda:embed(Xtr))
Ed,dev_embed_s,_=measure(lambda:embed(Xdev))
t0=time.perf_counter()
minilm_scores=[]
for candidate in [.1,1,10]:
    candidate_model=LogisticRegression(C=candidate,max_iter=1500,class_weight='balanced',random_state=SEED).fit(Etr,ytr)
    minilm_scores.append((f1_score(yd,candidate_model.predict(Ed),average='macro'),candidate))
c2=max(minilm_scores)[1]
tuning_records.append({'model':'Frozen MiniLM','tuning_s':dev_embed_s+(time.perf_counter()-t0),'selection':f'C={c2}','dev_macro_f1':max(minilm_scores)[0]})
m2,fit_s,_=measure(lambda:LogisticRegression(C=c2,max_iter=1500,class_weight='balanced',random_state=SEED).fit(Etr,ytr))
def predict_m2(x):return m2.predict(embed(x))
p,tinf,all_times=measure(lambda:predict_m2(Xte),repeats=5,warmup=1)
preds['Frozen MiniLM']=p
results['Frozen MiniLM']={**metrics(p),'train_s':train_embed_s+fit_s,'inference_s':tinf,'ms_per_sms':1000*tinf/len(Xte)}
timing_details['Frozen MiniLM']=all_times

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Method 3 — Fine-tuned DistilBERT

Development-set epoch selection (2 vs 3 epochs) is **tuning**, timed separately. Final training time includes **training-set tokenization plus model fine-tuning**. Test inference is timed using a direct PyTorch batch loop, including tokenization of raw SMS. This can take a long time on CPU. Do not run the test during tuning.

In [7]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,DataCollatorWithPadding
from datasets import Dataset
model_id='distilbert/distilbert-base-uncased'
tok=AutoTokenizer.from_pretrained(model_id)

def dataset(x,y):
    return Dataset.from_dict({'text':list(x),'labels':list(map(int,y))}).map(
        lambda b:tok(b['text'],truncation=True,max_length=128),batched=True,remove_columns=['text'])

def trainer_for(epochs,path,training_ds,dev_ds):
    args=TrainingArguments(output_dir=path,num_train_epochs=epochs,per_device_train_batch_size=16,
        per_device_eval_batch_size=32,learning_rate=2e-5,seed=SEED,data_seed=SEED,
        save_strategy='no',eval_strategy='no',report_to='none',dataloader_num_workers=0)
    model=AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=2)
    return Trainer(model=model,args=args,train_dataset=training_ds,eval_dataset=dev_ds,
        processing_class=tok,data_collator=DataCollatorWithPadding(tok))

# Tokenize train/dev once for tuning; account for train tokenization again in final training.
dtr,tok_train_s,_=measure(lambda:dataset(Xtr,ytr))
dd,tok_dev_s,_=measure(lambda:dataset(Xdev,yd))
tune_start=time.perf_counter()
tuning=[]
for epochs in [2,3]:
    trial=trainer_for(epochs,f'./tune_{epochs}',dtr,dd)
    trial.train()
    dev_p=np.argmax(trial.predict(dd).predictions,axis=1)
    tuning.append((f1_score(yd,dev_p,average='macro'),epochs))
    del trial
    if torch.cuda.is_available():torch.cuda.empty_cache()
best_epochs=max(tuning)[1]
tuning_records.append({'model':'Fine-tuned DistilBERT','tuning_s':time.perf_counter()-tune_start+tok_train_s+tok_dev_s,
    'selection':f'epochs={best_epochs}','dev_macro_f1':max(tuning)[0]})

# Final model: a new model trained on train only. Include train-tokenization time.
def train_final():
    final_ds=dataset(Xtr,ytr)
    final_trainer=trainer_for(best_epochs,'./final_model',final_ds,dd)
    final_trainer.train()
    return final_trainer
tr,final_train_s,_=measure(train_final)
tr.model.to(DEVICE).eval()

def predict_distilbert_raw(texts,batch_size=32):
    predictions=[]
    tr.model.eval()
    with torch.inference_mode():
        for i in range(0,len(texts),batch_size):
            batch=tok(texts[i:i+batch_size],return_tensors='pt',padding=True,
                truncation=True,max_length=128)
            batch={k:v.to(DEVICE) for k,v in batch.items()}
            predictions.extend(tr.model(**batch).logits.argmax(dim=-1).cpu().tolist())
    return np.asarray(predictions,dtype=int)

p,tinf,all_times=measure(lambda:predict_distilbert_raw(Xte),repeats=5,warmup=1)
preds['Fine-tuned DistilBERT']=p
results['Fine-tuned DistilBERT']={**metrics(p),'train_s':final_train_s,'inference_s':tinf,
    'ms_per_sms':1000*tinf/len(Xte)}
timing_details['Fine-tuned DistilBERT']=all_times

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3610 [00:00<?, ? examples/s]

Map:   0%|          | 0/774 [00:00<?, ? examples/s]

... (4 outputs left out)

Step,Training Loss
500,0.050400


Map:   0%|          | 0/3610 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.050700


In [8]:
# The table is populated ONLY after running the new timing protocol.
summary=pd.DataFrame(results).T
summary['tuning_s']=[next(r['tuning_s'] for r in tuning_records if r['model']==name) for name in summary.index]
display(summary.round(4))
print('Training includes feature extraction / training embeddings / training tokenization as applicable.')
print('Inference includes raw-text preprocessing and model prediction, median of 5 runs after 1 warm-up.')

,accuracy,macro_f1,spam_recall,train_s,inference_s,ms_per_sms,tuning_s
TF-IDF,0.9845,0.9646,0.9479,0.2002,0.0324,0.0419,0.6655
Frozen MiniLM,0.9845,0.9653,0.9688,4.0257,0.8097,1.0461,2.1117
Fine-tuned DistilBERT,0.9922,0.9820,0.9583,67.2037,1.7368,2.2440,123.8417


Training includes feature extraction / training embeddings / training tokenization as applicable.
Inference includes raw-text preprocessing and model prediction, median of 5 runs after 1 warm-up.


## Results, three errors, and secondary naturally occurring subsets

In [9]:
display(summary.round(4))
summary.to_csv('results_consistent_timing.csv')
pd.DataFrame(tuning_records).to_csv('tuning_costs.csv',index=False)
with open('timing_protocol_hardware.json','w') as f:
    json.dump({'hardware':HARDWARE,'timing':{'inference_repeats':5,'warmup':1,'batch_size_neural':32,
        'max_length_distilbert':128,'training':'raw train text through fitted model; exclude downloads/model loading',
        'inference':'raw test text through predictions; exclude model loading/metrics',
        'reported_statistic':'median seconds for full test batch; milliseconds per SMS = median*1000/test_size',
        'tuning':'separate from final train time'},'raw_inference_times_s':timing_details,
        'splits':{'train':len(Xtr),'dev':len(Xdev),'test':len(Xte)}},f,indent=2)
for name,p in preds.items():
    print('\n',name,'\n',classification_report(yt,p,target_names=['ham','spam'],zero_division=0))
    print('Confusion matrix:',confusion_matrix(yt,p).tolist())
    ids=np.where(p!=yt)[0][:5]
    display(pd.DataFrame({'text':np.array(Xte)[ids],'actual':yt[ids],'predicted':p[ids]}))
import re
pattern=r'(?i)\b(?:u|ur|txt|msg|2day|4u|2moro)\b'
mask=test.text.str.contains(pattern,regex=True).to_numpy()
print('Naturally occurring text-speak subset size:',int(mask.sum()),'of',len(yt))
print('Full cleaned corpus text-speak count:',int(df.text.str.contains(pattern,regex=True).sum()))
for name,p in preds.items():
    if mask.sum():
        print(name,'subset accuracy',round(accuracy_score(yt[mask],p[mask]),4),
              'subset spam count',int(yt[mask].sum()))
print('Exploratory subset only; regex is a heuristic and does not prove general robustness.')

,accuracy,macro_f1,spam_recall,train_s,inference_s,ms_per_sms,tuning_s
TF-IDF,0.9845,0.9646,0.9479,0.2002,0.0324,0.0419,0.6655
Frozen MiniLM,0.9845,0.9653,0.9688,4.0257,0.8097,1.0461,2.1117
Fine-tuned DistilBERT,0.9922,0.9820,0.9583,67.2037,1.7368,2.2440,123.8417



 TF-IDF 
               precision    recall  f1-score   support

         ham       0.99      0.99      0.99       678
        spam       0.93      0.95      0.94        96

    accuracy                           0.98       774
   macro avg       0.96      0.97      0.96       774
weighted avg       0.98      0.98      0.98       774

Confusion matrix: [[671, 7], [5, 91]]


,text,actual,predicted
0,I'm at work. Please call,0,1
1,K k:) sms chat with me.,0,1
2,Ok... Ur typical reply...,0,1
3,Have you laid your airtel line to rest?,0,1
4,Can U get 2 phone NOW? I wanna chat 2 set up m...,1,0



 Frozen MiniLM 
               precision    recall  f1-score   support

         ham       1.00      0.99      0.99       678
        spam       0.91      0.97      0.94        96

    accuracy                           0.98       774
   macro avg       0.95      0.98      0.97       774
weighted avg       0.99      0.98      0.98       774

Confusion matrix: [[669, 9], [3, 93]]


,text,actual,predicted
0,i am seeking a lady in the street and a freak ...,0,1
1,I've sent my wife your text. After we buy them...,0,1
2,"sir, you will receive the account no another 1...",0,1
3,Unlimited texts. Limited minutes.,0,1
4,"Aight I've been set free, think you could text...",0,1



 Fine-tuned DistilBERT 
               precision    recall  f1-score   support

         ham       0.99      1.00      1.00       678
        spam       0.98      0.96      0.97        96

    accuracy                           0.99       774
   macro avg       0.99      0.98      0.98       774
weighted avg       0.99      0.99      0.99       774

Confusion matrix: [[676, 2], [4, 92]]


,text,actual,predicted
0,TheMob>Yo yo yo-Here comes a new selection of ...,1,0
1,Im in inperialmusic listening2the weirdest tra...,0,1
2,Xmas & New Years Eve tickets are now on sale f...,1,0
3,Guess who am I?This is the first time I create...,1,0
4,Total video converter free download type this ...,0,1


Naturally occurring text-speak subset size: 159 of 774
Full cleaned corpus text-speak count: 1028
TF-IDF subset accuracy 0.9748 subset spam count 45
Frozen MiniLM subset accuracy 0.9937 subset spam count 45
Fine-tuned DistilBERT subset accuracy 0.9937 subset spam count 45
Exploratory subset only; regex is a heuristic and does not prove general robustness.


## Try it — new text without retraining

In [10]:
def try_it(message,method='Fine-tuned DistilBERT'):
    if not isinstance(message,str) or not message.strip():return 'Please enter a non-empty SMS message.'
    if method=='TF-IDF':p=int(m1.predict([message])[0])
    elif method=='Frozen MiniLM':p=int(predict_m2([message])[0])
    elif method=='Fine-tuned DistilBERT':p=int(predict_distilbert_raw([message])[0])
    else:return 'Unknown model'
    return 'Spam' if p else 'Not spam'
print(try_it('Are you free tonight?'))
print(try_it(''))

Not spam
Please enter a non-empty SMS message.


In [11]:
def try_it(message,method='Fine-tuned DistilBERT'):
    if not isinstance(message,str) or not message.strip():return 'Please enter a non-empty SMS message.'
    if method=='TF-IDF':p=int(m1.predict([message])[0])
    elif method=='Frozen MiniLM':p=int(predict_m2([message])[0])
    elif method=='Fine-tuned DistilBERT':p=int(predict_distilbert_raw([message])[0])
    else:return 'Unknown model'
    return 'Spam' if p else 'Not spam'
# Example 1 — Spam
spam_msg = "Congratulations! You've been selected to receive a free shopping voucher. Click the link and enter your card details to claim it."

# Example 2 — Not spam
ham_msg = "Hey! Are you free after class? I won a ticket for the event and wanted to invite you."

print("Example 1 — Expected: Spam")
print("Predicted:", try_it(spam_msg))

print("\nExample 2 — Expected: Not spam")
print("Predicted:", try_it(ham_msg))

Example 1 — Expected: Spam
Predicted: Spam

Example 2 — Expected: Not spam
Predicted: Not spam


## Save and reload models (reproducibility)

Run this after training. Reload the saved artifacts without retraining, and confirm predictions on a sample. Model files are large; upload them separately if the submission platform has size limits.

In [12]:

# Save and reload all three trained models
# Verify predictions on the same 10 held-out SMS messages

import joblib
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer
)

# 1. Create folder for saved models
artifact_dir = Path("group9_saved_models")
artifact_dir.mkdir(exist_ok=True)

# 2. Save all three models
joblib.dump(m1, artifact_dir / "tfidf_pipeline.joblib")
joblib.dump(m2, artifact_dir / "minilm_logreg.joblib")

enc.save(str(artifact_dir / "minilm_encoder"))

tr.model.save_pretrained(str(artifact_dir / "distilbert"))
tok.save_pretrained(str(artifact_dir / "distilbert"))

print("All three models saved successfully.")

# 3. Reload all three models
loaded_tfidf = joblib.load(
    artifact_dir / "tfidf_pipeline.joblib"
)

loaded_minilm_lr = joblib.load(
    artifact_dir / "minilm_logreg.joblib"
)

loaded_minilm = SentenceTransformer(
    str(artifact_dir / "minilm_encoder"),
    device=DEVICE
)

loaded_distilbert = (
    AutoModelForSequenceClassification.from_pretrained(
        str(artifact_dir / "distilbert")
    )
    .to(DEVICE)
    .eval()
)

loaded_tok = AutoTokenizer.from_pretrained(
    str(artifact_dir / "distilbert")
)

print("All three models reloaded successfully.")

# 4. Select the same 10 test messages
n_check = 10
test_messages = Xte[:n_check]

# 5. Verify TF-IDF predictions
tfidf_reloaded_preds = loaded_tfidf.predict(test_messages)

assert np.array_equal(
    tfidf_reloaded_preds,
    preds["TF-IDF"][:n_check]
)

print("TF-IDF verification passed.")

# 6. Verify MiniLM predictions
minilm_embeddings = loaded_minilm.encode(
    test_messages,
    batch_size=32,
    show_progress_bar=False
)

minilm_reloaded_preds = loaded_minilm_lr.predict(
    minilm_embeddings
)

assert np.array_equal(
    minilm_reloaded_preds,
    preds["Frozen MiniLM"][:n_check]
)

print("MiniLM verification passed.")

# 7. Verify DistilBERT predictions
distilbert_reloaded_preds = []

loaded_distilbert.eval()

with torch.inference_mode():
    for i in range(0, n_check, 32):

        batch_texts = test_messages[
            i:min(i + 32, n_check)
        ]

        batch = loaded_tok(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        outputs = loaded_distilbert(**batch)

        batch_preds = (
            outputs.logits
            .argmax(dim=-1)
            .cpu()
            .tolist()
        )

        distilbert_reloaded_preds.extend(batch_preds)

original_preds = np.asarray(
    preds["Fine-tuned DistilBERT"][:n_check]
)

reloaded_preds = np.asarray(
    distilbert_reloaded_preds
)

print("Original DistilBERT predictions:", original_preds)
print("Reloaded DistilBERT predictions:", reloaded_preds)

assert np.array_equal(
    original_preds,
    reloaded_preds
), "DistilBERT predictions do not match after reloading."

print("DistilBERT verification passed.")

# 8. Final confirmation
print(
    "SUCCESS: All 3 saved models were reloaded "
    "and matched predictions on 10 held-out messages."
)


All three models saved successfully.
All three models reloaded successfully.
TF-IDF verification passed.
MiniLM verification passed.
Original DistilBERT predictions: [0 0 0 1 0 0 0 0 0 0]
Reloaded DistilBERT predictions: [0 0 0 1 0 0 0 0 0 0]
DistilBERT verification passed.
SUCCESS: All 3 saved models were reloaded and matched predictions on 10 held-out messages.


## Sources
UCI SMS Spam Collection; Sentence-Transformers MiniLM; Hugging Face DistilBERT; scikit-learn. Cite original papers from the proposal. Restart and Run All before submission; preserve output cells and verify reproducibility.

**Report:** paste the newly generated `results_consistent_timing.csv` and hardware/timing methodology from `timing_protocol_hardware.json`; never reuse old timing figures. Include 3+ concrete misclassifications, natural text-speak subset counts and limitations, model-loading evidence, and all members’ original examples and appendix contributions. Instructor approval does not guarantee marks.